# Archetype Framework User Manual v2.0

## Table of Contents
1. [Introduction](#introduction)
2. [Understanding the Framework](#understanding-the-framework)
3. [Quick Start: Running Your First Archetype](#quick-start)
4. [Customizing Archetypes](#customizing-archetypes)
5. [Reference](#reference)

---

## Introduction

This document supports users of the Industrial Systems Analysis Lab's archetypes framework (https://github.com/ISALGroup/Industrial-decarb). The purpose is to make simplified models of industrial processes. This framework is inspired by Harry L. Brown's *Energy Analysis of 108 Industrial Processes*, which presented simplified process flowsheets with mass and energy balances. The Archetype Framework makes these models interactive and customizable.

### What is the Archetype Framework?

The Archetype Framework is a Python-based tool for modeling industrial processes. It allows you to:

1. **Run pre-built process models** (archetypes) for various industrial processes
2. **Customize process parameters** to match your specific scenarios
3. **Analyze energy and material flows** through industrial systems
4. **Compare different process configurations** and efficiency improvements

### Prerequisites

- **Python knowledge**: Basic familiarity with Python (variables, lists, functions, dictionaries)
- **Process understanding**: Basic knowledge of the industrial process you're modeling
- **Required Python packages**: pandas, numpy, csv (all standard Python libraries)

### Initialization
You can initialize the models with this code chunk (with archetypes_base in your file directory)


In [1]:
import csv
allflows = []
processunits = []
from archetypes_base import *


## Understanding the Framework

### Core Concepts

The framework models industrial processes using two main objects: **Process Flows** and **Process Units**.

Think of a process flowsheet as a network where:
- **Process Flows** are the pipes carrying material and energy
- **Process Units** are the equipment that transforms those flows

```
[Input Flow] → [Unit 1] → [Intermediate Flow] → [Unit 2] → [Output Flow]
```

### 1. Process Flows

**Flows** represent materials or energy moving through your process. A flow is like a pipe carrying material with the following properties:

#### Flow Attributes

All flow properties are stored in a dictionary called `flow.attributes`. You can access them from a flow like:

In [2]:
MyFlow = Flow()
flow_amount = MyFlow.attributes['mass_flow_rate']
flow_temp = MyFlow.attributes['temperature']

**Note:** These attribute dictionaries follow the same structure across all archetype models, though different archetypes may use different component names (e.g., 'Dry wood' vs 'Cellulose').

| Attribute | Type | Unit | Default | Description |
|-----------|------|------|---------|-------------|
| `name` | string | - | `""` | Unique identifier for the flow. Must be unique across all flows. For flows used in many units (steam, electricity, water), add the unit name to avoid confusion (e.g., 'Electricity (debarker)'). |
| `components` | list of strings | - | `""` | Names of components in the flow (e.g., `['Water', 'Ethanol']`). Use `None` for energy-only flows like electricity. |
| `flow_type` | string | - | `""` | Category of flow (see [Flow Types](#flow-types)). Used for energy/utility analysis and emissions handling. |
| `temperature` | int/float | °C | `25` | Temperature of the flow. |
| `pressure` | float | atm | `1` | Pressure of the flow. |
| `composition` | list of floats | - | `[1]` | Mass fraction of each component. Must match length of `components` and sum to 1.0. Example: 20% ethanol and 80% water = `[0.8, 0.2]` with components `['Water', 'Ethanol']`. |
| `origin` | string or None | - | `None` | Name of unit this flow comes from. Set to `None` for external inputs. |
| `destination` | string or None | - | `None` | Name of unit this flow goes to. Set to `None` for external outputs. |
| `mass_flow_rate` | int/float | kg/hr | `0` | Mass throughput rate of the flow. |
| `elec_flow_rate` | int/float | kW | `0` | Electrical power consumption/production. |
| `heat_flow_rate` | int/float | kJ/hr | `0` | Heat content above ambient temperature. Calculated as the difference between current enthalpy and enthalpy at ambient (liquid water at ambient = 0). |
| `combustion_energy_content` | int/float | kJ/hr | `0` | Gross heat of combustion if flow were fully combusted. |

#### Additional Flow Properties

These properties are NOT stored in the `attributes` dictionary, but as direct attributes of the Flow object:

| Property | Type | Default | Description |
|----------|------|---------|-------------|
| `is_calc_flow` | boolean | `False` | If `True`, this flow is a reference flow used as the basis for unit calculations. |
| `is_shear_stream` | boolean | `False` | If `True`, this flow creates a recycle loop in the process. |

#### What is a Reference Flow?

A **reference flow** (also called a **calc_flow** or **calculation flow**) is a flow whose properties are already known and serves as the basis for calculating other flows in a unit.

**Key principle:** Every unit needs at least one reference flow to drive its calculations.

**How to mark a flow as a reference flow:**

In [4]:
# Method 1: During initialization
MyFlow = Flow(name='Logs', is_calc_flow=True)

# Method 2: After creation
MyFlow = Flow(name='Logs',)
MyFlow.set_calc_flow()

**Example:** In a debarker, the incoming logs are the reference flow. You know:
- 1000 kg/hr of logs enter
- Composition: 35% wood, 5% bark, 60% water

The debarker uses this known input to calculate:
- How much electricity is needed (8 kWh per ton)
- How much bark comes out (125 kg/hr)
- How much debarked wood comes out (875 kg/hr)

The debarked logs output can then become the reference flow for the next unit (chipper), and so on through the process.

#### Understanding Composition

Composition describes what's in a flow as mass fractions that sum to 1.0.

**Example: Logs with 60% moisture, 1/8 of dry matter is bark**

In [5]:
# Given:
total_mass = 1000  # kg
moisture_fraction = 0.60

# Step 1: Calculate water and dry matter
water_mass = 1000 * 0.60  # = 600 kg
dry_mass = 1000 * 0.40    # = 400 kg

# Step 2: Split dry matter (1/8 bark, 7/8 wood)
bark_fraction_of_dry = 1/8  # = 0.125
bark_mass = 400 * 0.125     # = 50 kg
wood_mass = 400 * 0.875     # = 350 kg

# Step 3: Express as fractions of total mass
components = ['Dry wood', 'Dry bark', 'Water']
composition = [
    350/1000,  # 0.35 (35% wood)
    50/1000,   # 0.05 (5% bark)
    600/1000   # 0.60 (60% water)
]
# composition = [0.35, 0.05, 0.60]  ✓ Sums to 1.0

#### Flow Functions

The Flow class contains methods for connecting flows to units and setting properties:

| Function | Purpose | Example |
|----------|---------|---------|
| `set_destination(Unit)` | Set destination to Unit and add flow to Unit's input_flows list | `MyFlow.set_destination(Debarker)` |
| `set_origin(Unit)` | Set origin to Unit and add flow to Unit's output_flows list | `MyFlow.set_origin(Debarker)` |
| `set_calc_flow()` | Mark this flow as a reference flow (sets `is_calc_flow = True`) | `MyFlow.set_calc_flow()` |
| `set_shear_stream()` | Mark this flow as a recycle stream (sets `is_shear_stream = True`) | `MyFlow.set_shear_stream()` |
| `detach_flow(Unit)` | Remove connection to Unit (sets origin/destination to None) | `MyFlow.detach_flow(Debarker)` |

#### Flow Example: Creating a Log Stream

Imagine a log stream being transported to a debarking unit. One ton gets processed every hour, the logs are at 60% moisture, and 1/8 of the dry matter is bark. Here's how to set it up:

In [6]:
MyFlow = Flow(
    name='Logs',
    components=['Dry wood', 'Dry bark', 'Water'],
    flow_type='Input flow',
    temperature=25,
    pressure=1,
    composition=[0.35, 0.05, 0.6],  # 35% wood, 5% bark, 60% water
    origin=None,  # External input
    destination=None,  # Will be set to 'Debarker' later
    mass_flow_rate=1000,  # kg/hr
    elec_flow_rate=0,
    heat_flow_rate=0,
    combustion_energy_content=0,  # Can be changed if wood/bark will be burned
    is_calc_flow=True,  # This is the reference flow
    is_shear_stream=False
)

### 2. Process Units

**Units** represent equipment that transforms flows.

#### Unit Attributes

| Attribute | Type | Default | Description |
|-----------|------|---------|-------------|
| `name` | string | (required) | Unique identifier for the unit. Used in flow `origin` and `destination`. Recommended to use unique names with numbering if multiple units of same type exist. |
| `input_flows` | list of strings | `[]` | Names of flows currently connected as inputs. **Managed automatically by framework.** |
| `output_flows` | list of strings | `[]` | Names of flows currently connected as outputs. **Managed automatically by framework.** |
| `expected_flows_in` | list of strings | `[]` | Names of flows that SHOULD connect as inputs. **You define this.** |
| `expected_flows_out` | list of strings | `[]` | Names of flows that SHOULD connect as outputs. **You define this.** |
| `calculations` | dict or tuple | `{}` | Calculation function(s).  See [Calculations Format](#calculations-format) below. |
| `coefficients` | dict | `{}` | Unit parameters (efficiency, power consumption, ratios, etc.). **These are what you customize!** |
| `unit_type` | string | `'generic'` | Category of unit (optional classification). |
| `tags` | list of strings | `[]` | User-defined tags for classification. |
| `temperature` | float or None | `None` | Operating temperature of unit (°C). |
| `is_calc` | boolean | `False` | `True` if unit has been calculated. **Managed automatically.** |
| `required_calc_flows` | int | `1` | Number of reference flows needed before unit can calculate. |

**Important distinction:**
- `expected_flows_in` / `expected_flows_out`: What SHOULD be connected (you define this when creating the unit)
- `input_flows` / `output_flows`: What IS currently connected (managed automatically by the framework)

#### Calculations Format

**Single reference flow (most common):**
If only one calc flow is required: A dictionary of flow names (strings) as keys that are used as the reference flow for calculations and function as values that will be used to calculate subsequent flows attached to the unit. 

Generic syntax (Don't run, this just shows the pattern) 


In [ ]:
MyUnit.calculations = {'Reference Flow Name': your_calculation_function}

**Multiple reference flows:**
If multuple calc_flows are reqiured: a tuple list of flow names used as reference flows for calculations and a function that uses the attributes of those flows.

Generic syntax (again just showing the pattern):

In [ ]:
MyUnit.calculations = (['Flow 1', 'Flow 2'], your_calculation_function)
MyUnit.required_calc_flows = 2

#### Unit Functions

| Function | Purpose | Returns |
|----------|---------|---------|
| `is_fully_calc()` | Check if all expected flows are attached | `True` or `False` |
| `set_flow(flow_dict, flowlist)` | Create flow from dictionary and attach to unit | None |
| `count_calc_flows(flowlist)` | Count reference flows attached to unit | int |
| `custom_str(flowlist)` | Get detailed string representation with flow info | string |
| `calc(flowlist, unitlist)` | Execute calculation function and create output flows | None |
| `attach_available_flow(flowlist)` | Attach expected flows from flowlist to this unit | None |
| `are_all_flows_attached()` | Check if all expected flows are in input/output lists | `True` or `False` |
| `check_mass_balance(flowlist)` | Verify mass in = mass out (within 0.1% tolerance) | `True` or `False` |
| `check_heat_balance(flowlist)` | Verify energy in = energy out (within 0.1% tolerance) | `True` or `False` |

#### Unit Example: Creating a Debarker

A debarker takes in logs with bark and electricity to mechanically tumble them together, using friction between logs to remove bark. We assume:
- Total separation of bark and wood
- Moisture is the same in bark and wood
- Debarker needs 8 kWh per ton of wood

**Step 1: Define the unit structure**

In [11]:
Debarker = Unit('Debarker')
Debarker.expected_flows_in = ['Logs', 'Electricity (debarker)']
Debarker.expected_flows_out = ['Bark', 'Debarked logs']
Debarker.coefficients = {'Power per t log': 8}

**Step 2: Define the calculation function**

The function takes a reference flow (logs) and coefficients, then calculates outputs:

In [12]:
def Debarkfunc_logs(log_flow, coefficients):
    # Extract data from reference flow
    log_amount = log_flow.attributes['mass_flow_rate']
    electricity_amount = coefficients['Power per t log'] * log_amount/1000.
    # Note: Divide by 1000 because mass_flow_rate is in kg but coefficient is per ton
    
    # Get component indices from the reference flow
    bark_index = log_flow.attributes['components'].index('Dry bark')
    wood_index = log_flow.attributes['components'].index('Dry wood')
    moisture_index = log_flow.attributes['components'].index('Water')
    
    # Get composition ratios
    bark_ratio = log_flow.attributes['composition'][bark_index]
    wood_ratio = log_flow.attributes['composition'][wood_index]
    moisture_ratio = log_flow.attributes['composition'][moisture_index]
    
    # Calculate how to split the logs
    dry_bark_ratio = bark_ratio/(bark_ratio + wood_ratio)
    dry_wood_ratio = wood_ratio/(bark_ratio + wood_ratio)
    wood_flow_amount = dry_wood_ratio * log_amount
    bark_flow_amount = dry_bark_ratio * log_amount
    
    # Note: We assume ambient temperature operation, so heat_flow_rate = 0
    
    # Return list of output flows as dictionaries
    return [
        {
            'name': 'Electricity (debarker)',
            'components': None,
            'mass_flow_rate': 0,
            'flow_type': 'Electricity',
            'elec_flow_rate': electricity_amount,
            'In or out': 'In',
            'Set calc': False,
            'Set shear': False
        },
        {
            'name': 'Bark',
            'components': ['Dry bark', 'Water'],
            'composition': [1-moisture_ratio, moisture_ratio],
            'mass_flow_rate': bark_flow_amount,
            'flow_type': 'Process stream',
            'temperature': 25,
            'pressure': 1,
            'In or out': 'Out',
            'Set calc': True,  # Could be reference flow for next unit
            'Set shear': False
        },
        {
            'name': 'Debarked logs',
            'components': ['Dry wood', 'Water'],
            'composition': [1-moisture_ratio, moisture_ratio],
            'mass_flow_rate': wood_flow_amount,
            'flow_type': 'Process stream',
            'temperature': 25,
            'pressure': 1,
            'In or out': 'Out',
            'Set calc': True,  # WILL be reference flow for chipper unit
            'Set shear': False
        }
    ]

**Step 3: Assign the calculation function**

In [13]:
Debarker.calculations = {'Logs': Debarkfunc_logs}

The key `'Logs'` tells the framework which flow is the reference flow for this unit.

**Step 4: Add to lists and run**

In [14]:
# Add to process lists
processunits.append(Debarker)
allflows.append(MyFlow)  # The Logs flow created earlier

# Attach flows and calculate
processunits[0].attach_available_flow(allflows)
processunits[0].calc(allflows, processunits)

# Print results
for flow in allflows:
    print(flow)

Flow Logs . Type :Input flow. 
Components: ['Dry wood', 'Dry bark', 'Water']with composition [0.35, 0.05, 0.6]. 
 No origin. Destination:Debarker . 
Mass flow rate: 1000kg/hr. Heat flow rate: 0kJ/hr . 
Electricity flow rate : 0kW.  Combustion energy contained: 0kJ / hr.
 

Flow Electricity (debarker) . Type :Electricity. 
Components: Nonewith composition [1]. 
 No origin. Destination:Debarker . 
Mass flow rate: 0kg/hr. Heat flow rate: 0kJ/hr . 
Electricity flow rate : 8.0kW.  Combustion energy contained: 0kJ / hr.
 

Flow Bark . Type :Process stream. 
Components: ['Dry bark', 'Water']with composition [0.4, 0.6]. 
Origin : Debarker , no destination. 
 Mass flow rate: 125.00000000000003kg/hr. Heat flow rate: 0kJ/hr . 
Electricity flow rate : 0kW.  Combustion energy contained: 0kJ / hr.
 

Flow Debarked logs . Type :Process stream. 
Components: ['Dry wood', 'Water']with composition [0.4, 0.6]. 
Origin : Debarker , no destination. 
 Mass flow rate: 875.0kg/hr. Heat flow rate: 0kJ/hr . 
Ele

**How calculations propagate through the process:**

```
Logs (1000 kg/hr, is_calc_flow=True)
    ↓
[Debarker calculation function runs]
    ↓
Creates: Electricity (8 kW), Bark (125 kg/hr), Debarked Logs (875 kg/hr, Set calc=True)
    ↓
Debarked Logs becomes reference flow for Chipper unit
    ↓
[Chipper calculation function runs]
    ↓
And so on through the process...
```

### Flow Types

Flow types categorize flows for energy and material analysis:

**Material Flows:**
- `'Input flow'`: Raw materials entering the process
- `'Process flow'` / `'Process stream'`: Intermediate materials between units
- `'Product'`: Final product leaving the process

**Energy Flows:**
- `'Electricity'`: Electrical power consumption (kW)
- `'Steam'`: Steam for heating (with heat_flow_rate)
- `'Fuel'`: Combustible fuel for burning
- `'Steam (produced on-site)'`: Steam generated within the process
- `'Fuel (produced on-site)'`: Fuel/combustibles generated as byproduct

**Utility Flows:**
- `'Water'`: Process water input
- `'Compressed air'`: Compressed air for processes
- `'Compressed air (produced on-site)'`: Air compressed on-site

**Waste Flows:**
- `'Waste'`: Solid waste streams
- `'Waste water'` / `'Wastewater'`: Liquid waste
- `'Exhaust'`: Exhaust gases leaving the process
- `'Condensate'`: Condensed steam returning from process
- `'Hot water return'`: Hot water returning for reuse

### Key Terminology Summary

| Term | Meaning | When to Use |
|------|---------|-------------|
| **Reference flow / calc_flow** | A flow with known properties used as the basis for calculating other flows | Set `is_calc_flow=True` for input flows or flows whose properties you know |
| **Shear stream** | A flow that creates a loop/recycle in the process | Set `is_shear_stream=True` for streams like white liquor that recycle back |
| **In or out** | Direction of flow relative to unit | `'In'` for inputs, `'Out'` for outputs (REQUIRED in calculation returns) |
| **Set calc** | Whether this output flow should be used as a reference flow by downstream units | `True` if other units will base calculations on this flow |
| **Set shear** | Whether this is a recycle/loop flow | `True` for recycle streams, `False` for once-through flows |
| **Coefficients** | Unit-specific parameters that control behavior | Store constants like efficiency, power consumption, ratios |
| **expected_flows** | List of flow names that should connect to unit | You define this structure |
| **input_flows / output_flows** | List of flow names currently connected | Framework manages this automatically |

---

## Quick Start: Running Your First Archetype (2 minutes)

### Step 1: Run an Existing Archetype 

In [15]:
# Import required libraries
import pandas as pd
import numpy as np
import csv
from archetypes_base import *

# Load and run an archetype
exec(open('archetype_bleached_kraft_integrated.py').read())

# The model is now calculated! Check results:
print(f"\nTotal heat demand: {calc_heat_demand(allflows, processunits):,.0f} kJ/hr")
print(f"Number of units: {len(processunits)}")
print(f"Number of flows: {len(allflows)}")

Presence of shear streamWhite liquor. Original mass flow rate =999.9999999999999 kg/hr. New mass flow rate = 1000.0 kg/hr. Relative difference = -1.1368683772161603e-16. Original heat flow rate =85274.99999999999 kJ/hr. New heat flow rate = 85275.0 kJ/hr. Relative difference = -1.7064690974338146e-16
{'Electricity per ton of CaCO3': 60, 'Quicklime temperature': 650, 'Stack temperature': 400, 'Air ratio': 1, 'Losses': 0.1}
1675.2675952251957
84.0
10.237500000000002
66083.15493651654
28508.778000000006
94591.93293651655
103345.44760143485
10334.544760143486
Presence of shear streamQuicklime. Original mass flow rate =28.0 kg/hr. New mass flow rate = 0.028000000000000004 kg/hr. Relative difference = 0.999. Original heat flow rate =30395.076923076922 kJ/hr. New heat flow rate = 10.237500000000002 kJ/hr. Relative difference = 0.9996631855867347
Flow Logs . Type :input. 
Components: ['Water', 'Bark', 'Wood']with composition [0.55, 0.06749999999999999, 0.38249999999999995]. 
 No origin. Destin

### Step 2: Export Results to CSV Files

In [16]:
# Export all results
flows_to_file('baseline_results', allflows)
unit_recap_to_file('baseline_results', allflows, processunits)
utilities_recap('baseline_results', allflows, processunits)

print("\nResults exported to:")
print("  - baseline_results_flows.csv")
print("  - baseline_results_units.csv")
print("  - baseline_results_utilites.csv")


Results exported to:
  - baseline_results_flows.csv
  - baseline_results_units.csv
  - baseline_results_utilites.csv


### Step 3: Explore the Results

Open the CSV files to see:
- **baseline_results_flows.csv**: All material and energy flows with compositions, temperatures, flow rates
- **baseline_results_units.csv**: Each unit with its input/output flows, heat losses, emissions
- **baseline_results_utilities.csv**: Energy demand and production by unit

**That's it!** You've run your first archetype. Now let's learn how to customize it.

---

## Customizing Archetypes

Archetypes are pre-built process models (like `archetype_bleached_kraft_integrated.py`) that represent complete industrial processes. You can customize them by changing parameters without rewriting code.

Every archetype has two types of customizable parameters:

1. **Global Variables**: Process-wide parameters at the top of the file
2. **Unit Coefficients**: Equipment-specific parameters in each unit's `coefficients` dictionary

### Step 1: Identify Customizable Parameters

Open the archetype file and look at the top for **global variables**. These control the overall process.

**Example from bleached kraft archetype:**

In [17]:
## Global variables
amb_t = 20  # Ambient temperature (°C)
recovery_yield = 0.5  # Pulp recovery yield (fraction)
NaOH_in_WL = 40  # NaOH in white liquor (g/kg)
Na2S_in_WL = 15.6  # Na2S in white liquor (g/kg)
wood_reference_flow = 1000  # kg of wood in digester - CONTROLS THROUGHPUT
bark_in = 0.15  # Bark fraction in input logs
bark_out = 0.01  # Bark remaining after debarking
wood_moisture = 0.55  # Moisture fraction in wood
heat_bark = 5900  # kJ/kg - Heating value of bark

### Step 2: Understand Unit Coefficients

Scroll through the archetype to find unit definitions. Each unit has a `coefficients` dictionary:

**Example: Debarker (Unit 1)**

In [19]:
Unit1.coefficients = {
    'Power per t log': 8.5,  # kWh per ton of logs
    'Bark out': bark_out  # Fraction of bark remaining
}

**Example: Bleaching (Unit 9)**

In [18]:
Unit9.coefficients = {
    'Bleaching cycles': 3,
    'Consistency': 0.1,
    'kg ClO2 per t pulp': 15,
    'kg O2 per t pulp': 10,
    'kg NaOH per t pulp': 20,
    'Bleaching rate': 0.1,
    'Electricity consumption per bleaching step and per t of pulp': 3,
    'Bleaching temperature': 100,
    'Losses': 0.1
}

### Step 3: Create a Custom Scenario

Create a new Python file to run customized scenarios:

In [19]:
# ========================================
# RE-RUN THE MODEL
# ========================================

# Reset flow list
allflows = []

# Reset units AND clear their flow lists
processunits = [Unit1, Unit2, Unit3, Unit4, Unit5, Unit6, Unit7, Unit8, 
                Unit9, Unit10, Unit11, Unit12, Unit13, Unit14, Unit15, 
                Unit16, Unit17, Unit18, Unit19, Unit20]

# IMPORTANT: Clear the input/output flows from previous run
for unit in processunits:
    unit.input_flows = []
    unit.output_flows = []
    unit.is_calc = False  # Also reset calculation status

# Recreate the initial flow with new parameters
FlowA = Flow('Logs', ['Water', 'Bark', 'Wood'], 'input', amb_t, 1, 
             [wood_moisture, normal_bark_in, 1 - wood_moisture - normal_bark_in],
             None, None, wood_flow_amount, np.nan, 0)
FlowA.set_calc_flow()
allflows.append(FlowA)

# Run the model
main(allflows, processunits, f_print=False)

Presence of shear streamWhite liquor. Original mass flow rate =999.9999999999999 kg/hr. New mass flow rate = 1000.0 kg/hr. Relative difference = -1.1368683772161603e-16. Original heat flow rate =85274.99999999999 kJ/hr. New heat flow rate = 85275.0 kJ/hr. Relative difference = -1.7064690974338146e-16
{'Electricity per ton of CaCO3': 60, 'Quicklime temperature': 650, 'Stack temperature': 400, 'Air ratio': 1, 'Losses': 0.1}
1675.2675952251957
84.0
10.237500000000002
66083.15493651654
28508.778000000006
94591.93293651655
103345.44760143485
10334.544760143486
Presence of shear streamQuicklime. Original mass flow rate =28.0 kg/hr. New mass flow rate = 0.028000000000000004 kg/hr. Relative difference = 0.999. Original heat flow rate =30395.076923076922 kJ/hr. New heat flow rate = 10.237500000000002 kJ/hr. Relative difference = 0.9996631855867347


([<archetypes_base.Flow at 0x10f1af410>,
  <archetypes_base.Unit at 0x10f1ffee0>])

### Step 4: Compare Multiple Scenarios

Run and compare baseline vs. improved scenarios:

In [20]:
# compare_scenarios.py
import pandas as pd
import numpy as np
import csv
from archetypes_base import *

# ========================================
# SCENARIO 1: BASELINE
# ========================================
print("Running Baseline Scenario...")
exec(open('archetype_bleached_kraft_integrated.py').read())
baseline_heat = calc_heat_demand(allflows, processunits)
flows_to_file('baseline', allflows)
utilities_recap('baseline', allflows, processunits)

# ========================================
# SCENARIO 2: IMPROVED EFFICIENCY
# ========================================
print("Running Improved Efficiency Scenario...")

# Modify parameters
recovery_yield = 0.55  # Better yield
Unit1.coefficients['Power per t log'] = 7.0  # More efficient
Unit9.coefficients['kg ClO2 per t pulp'] = 12  # Less chemical

# IMPORTANT: Reset units before re-running
allflows = []
processunits = [Unit1, Unit2, Unit3, Unit4, Unit5, Unit6, Unit7, Unit8, 
                Unit9, Unit10, Unit11, Unit12, Unit13, Unit14, Unit15, 
                Unit16, Unit17, Unit18, Unit19, Unit20]

# Clear the flow lists from previous run
for unit in processunits:
    unit.input_flows = []
    unit.output_flows = []
    unit.is_calc = False

# Recreate initial flow
FlowA = Flow('Logs', ['Water', 'Bark', 'Wood'], 'input', amb_t, 1, 
             [wood_moisture, normal_bark_in, 1 - wood_moisture - normal_bark_in],
             None, None, wood_flow_amount, np.nan, 0)
FlowA.set_calc_flow()
allflows.append(FlowA)
main(allflows, processunits, f_print=False)

improved_heat = calc_heat_demand(allflows, processunits)
flows_to_file('improved', allflows)
utilities_recap('improved', allflows, processunits)

# ========================================
# COMPARE RESULTS
# ========================================
print(f"\n{'='*50}")
print("COMPARISON SUMMARY")
print(f"{'='*50}")
print(f"Baseline heat demand:  {baseline_heat:,.0f} kJ/hr")
print(f"Improved heat demand:  {improved_heat:,.0f} kJ/hr")
print(f"Heat savings:          {baseline_heat - improved_heat:,.0f} kJ/hr")
print(f"Percent reduction:     {(baseline_heat - improved_heat)/baseline_heat * 100:.1f}%")

Running Baseline Scenario...
Presence of shear streamWhite liquor. Original mass flow rate =999.9999999999999 kg/hr. New mass flow rate = 1000.0 kg/hr. Relative difference = -1.1368683772161603e-16. Original heat flow rate =85274.99999999999 kJ/hr. New heat flow rate = 85275.0 kJ/hr. Relative difference = -1.7064690974338146e-16
{'Electricity per ton of CaCO3': 60, 'Quicklime temperature': 650, 'Stack temperature': 400, 'Air ratio': 1, 'Losses': 0.1}
1675.2675952251957
84.0
10.237500000000002
66083.15493651654
28508.778000000006
94591.93293651655
103345.44760143485
10334.544760143486
Presence of shear streamQuicklime. Original mass flow rate =28.0 kg/hr. New mass flow rate = 0.028000000000000004 kg/hr. Relative difference = 0.999. Original heat flow rate =30395.076923076922 kJ/hr. New heat flow rate = 10.237500000000002 kJ/hr. Relative difference = 0.9996631855867347
Flow Logs . Type :input. 
Components: ['Water', 'Bark', 'Wood']with composition [0.55, 0.06749999999999999, 0.3824999999

### Common Parameters to Customize

| Parameter Type | Example | Where to Find | Impact |
|----------------|---------|---------------|--------|
| **Throughput** | `wood_reference_flow = 2000` | Global variables | Scales entire process proportionally |
| **Recovery/Yield** | `recovery_yield = 0.55` | Global variables | Affects product yield and waste generation |
| **Equipment Efficiency** | `'Power per t log': 7.5` | Unit coefficients | Changes electricity demand per unit |
| **Chemical Usage** | `'kg ClO2 per t pulp': 12` | Unit coefficients | Affects chemical inputs and costs |
| **Operating Temperature** | `'Bleaching temperature': 95` | Unit coefficients | Changes heat/steam requirements |
| **Process Intensity** | `'Bleaching cycles': 4` | Unit coefficients | Affects quality, time, and energy use |
| **Heat Losses** | `'Losses': 0.08` | Unit coefficients | Changes energy efficiency |

### Tips for Customization

1. **Start small**: Change one parameter at a time to understand its impact
2. **Keep notes**: Document what you changed and why
3. **Use descriptive filenames**: Export each scenario with a clear name
4. **Check mass/energy balances**: After major changes, verify balances:
   ```python
   for unit in processunits:
       unit.check_mass_balance(allflows)
       unit.check_heat_balance(allflows)
   ```
4. **Look for connected parameters**: Some global variables are used in multiple units
5. **Physical limits**: Keep parameters realistic (e.g., recovery yield < 1.0)

---

## Reference

### Global Functions

| Function | Purpose | Parameters | Returns |
|----------|---------|------------|---------|
| `main(flowlist, unitlist, f_print)` | Run all calculations iteratively until complete | flowlist, unitlist, f_print=False | (flowlist, unitlist) |
| `flows_to_file(filename, flowlist)` | Export all flows to CSV file | filename (without .csv), flowlist | None |
| `unit_recap_to_file(filename, flowlist, unitlist)` | Export unit summary with connected flows to CSV | filename, flowlist, unitlist | None |
| `utilities_recap(filename, flowlist, unitlist)` | Export energy/utility analysis by unit to CSV | filename, flowlist, unitlist | None |
| `calc_heat_demand(flowlist, unitlist)` | Calculate total indirect heat demand of process | flowlist, unitlist | float (kJ/hr) |
| `find_Flow_index(name, flowlist)` | Get index of flow by name | name (string), flowlist | int or None |
| `flow_already_present(name, flowlist)` | Check if flow with name exists in flowlist | name (string), flowlist | `True` or `False` |
| `find_Unit_index(name, unitlist)` | Get index of unit by name | name (string), unitlist | int or None |
| `are_units_calced(unitlist)` | Check if all units are calculated | unitlist | `True` or `False` |
| `print_flows(flowlist)` | Print all flows to console | flowlist | None |

### Output Files

The framework can export results to CSV files:

#### flows_to_file()
Creates a CSV with all flow attributes (one row per flow):
- name, components, flow_type, temperature, pressure, composition
- origin, destination, mass_flow_rate, elec_flow_rate, heat_flow_rate, combustion_energy_content

**Example use:**

In [30]:
flows_to_file('my_results', allflows)
# Creates: my_results_flows.csv

#### unit_recap_to_file()
Creates a CSV with unit summaries including:
- Unit name
- Input flows (name, temperature, mass flow rate, heat flow rate, electricity)
- Output flows (name, temperature, mass flow rate, heat flow rate, electricity)
- Heat losses and reaction heats (if applicable)
- Emissions (greenhouse gases and copollutants, if applicable)

**Example use:**

In [31]:
unit_recap_to_file('my_results', allflows, processunits)
# Creates: my_results_units.csv

#### utilities_recap()
Creates a CSV with energy analysis by unit:
- Heat demand (steam and fuel, in kJ)
- Electricity demand (kWh)
- Heat and fuel produced on-site
- Electricity produced on-site
- Waste heat
- Compressed air demand and production

**Example use:**

In [32]:
utilities_recap('my_results', allflows, processunits)
# Creates: my_results_utilites.csv

### Advanced: Multiple Reference Flows

Some units need multiple input flows to calculate their outputs. For example, a mixer needs to know the flow rates of both input streams.

**Example: Merging two streams**

In [36]:
import pandas as pd
import numpy as np
import csv
import inspect
from archetypes_base import *
allflows = []
processunits = []

MultiplecalcflowUnit = Unit('Multiple calc')
MultiplecalcflowUnit.expected_flows_in = ['A', 'B']
MultiplecalcflowUnit.expected_flows_out = ['C']
MultiplecalcflowUnit.required_calc_flows = 2

def ABMerge(ablist, coeff):
    a_flow = ablist[0]
    b_flow = ablist[1]
    a_amount = a_flow.attributes['mass_flow_rate']    
    b_amount = b_flow.attributes['mass_flow_rate']
    c_amount = a_amount + b_amount
    return [{'name' : 'C', 'components' : None, 'mass_flow_rate' : c_amount,
             'flow_type': 'Product', 'elec_flow_rate' : 0 , 'In or out' : 'Out', 'Set calc' : False, 'Set shear' : False}, {'Heat of reaction': 1000} ]

MultiplecalcflowUnit.calculations = (['A', 'B'], ABMerge)

FlowA = Flow('A',[],'input', 25, 1, [1], None , None, 1000, np.nan, 0)
FlowA.set_calc_flow()
FlowB = Flow('B',[],'input', 25, 1, [1], None , None, 500, np.nan, 0)
FlowB.set_calc_flow()
allflows.append(FlowA)
allflows.append(FlowB)
processunits.append(MultiplecalcflowUnit)
processunits[0].attach_available_flow(allflows)
print(processunits[0].count_calc_flows(allflows))
processunits[0].calc(allflows, processunits)
print(processunits[0])

print_flows(allflows)
print(processunits[0].reaction_heat)

2
Unit Multiple calc
Input flows ['A', 'B']
Output flows ['C']
Expected input flows ['A', 'B']
Expected output flow ['C']

Flow A . Type :input. 
Components: []with composition [1]. 
 No origin. Destination:Multiple calc . 
Mass flow rate: 1000kg/hr. Heat flow rate: 0kJ/hr . 
Electricity flow rate : nankW.  Combustion energy contained: 0kJ / hr.
 

Flow B . Type :input. 
Components: []with composition [1]. 
 No origin. Destination:Multiple calc . 
Mass flow rate: 500kg/hr. Heat flow rate: 0kJ/hr . 
Electricity flow rate : nankW.  Combustion energy contained: 0kJ / hr.
 

Flow C . Type :Product. 
Components: Nonewith composition [1]. 
Origin : Multiple calc , no destination. 
 Mass flow rate: 1500kg/hr. Heat flow rate: 0kJ/hr . 
Electricity flow rate : 0kW.  Combustion energy contained: 0kJ / hr.
 

1000


### Advanced: Heat and Energy Balances

For units with heat transfer or chemical reactions, you can specify heat losses, heat of reaction, and emissions.

**Heat Loss Example:**

In [34]:
def calc_heater(ref_flow, coeff):
    # ... calculations ...
    Q_loss = Q_in * coeff['Loss fraction']
    
    return [
        # ... normal flow dictionaries ...
        {'Heat loss': Q_loss}  # Single-key dict for heat loss (kJ/hr)
    ]

**Heat of Reaction Example:**

In [ ]:
def calc_reactor(ref_flow, coeff):
    # ... calculations ...
    Q_reaction = mass_reacted * coeff['Heat of reaction per kg']
    
    return [
        # ... normal flow dictionaries ...
        {'Heat of reaction': Q_reaction}  # Positive = exothermic (kJ/hr)
    ]

**Emissions Example:**

In [35]:
def calc_combustor(ref_flow, coeff):
    # ... calculations ...
    co2_amount = carbon_in * (44/12)  # kg/hr
    
    return [
        # ... normal flow dictionaries ...
        {'Emissions': {
            'CO2': co2_amount,  # kg/hr
            'NOx': nox_amount,
            'SOx': sox_amount
        }}
    ]

The framework will include these in mass/energy balance checks and export them in the unit recap file.

## Version History

